## Ejemplo

- Organizar el código **optimizando la legibilidad y manteniendo siempre el nivel de complejidad de cada función por debajo de cierto umbral**. En el ejemplo, la complejidad se mide empleando la complejidad ciclomática y el umbral es 7.


- Preservar el nivel de abstracción: evitar mezclar, en la misma función, llamadas a funciones que indican *qué* hace una tarea con código que muestra *cómo* se realiza una tarea. **Los nombres de las funciones y variables deben reflejar su propósito**. A mayor cantidad de líneas en la función, más difícil encontrar un nombre adecuado.


- Aplicar el [**principio de responsabilidad única**](https://es.wikipedia.org/wiki/Principio_de_responsabilidad_%C3%BAnica): toda función debe tener solo una razón para ser modificada.


- Preferir un enfoque funcional (para evitar ciclos y estructuras condicionales lo más que se pueda).


In [35]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
import csv
import re
import requests
from bs4 import BeautifulSoup

In [37]:
def extraer_nombre(div_automovil):
    return div_automovil.find('span', class_='car_name').text

In [38]:
def extraer_cilindros(div_automovil):
    str_cilindros = div_automovil.find('span', class_='cylinders').text
    return int(str_cilindros)

In [39]:
def extraer_peso(div_automovil):
    str_peso = div_automovil.find('span', class_='weight').text
    return int(str_peso.replace(',', ''))

In [40]:
def extraer_aceleracion(div_automovil):
    return float(div_automovil.find('span', class_='acceleration').text)

In [41]:
#Funciones que realizan más de una tarea deben ser la excepción, NO la regla
def extraer_territorio_anio(div_automovil):
    str_from = div_automovil.find("span", attrs={"class":"from"}).text
    anio, territorio = str_from.strip('()').split(',')
    anio = int(anio.strip())
    territorio = territorio.strip()
    return territorio, anio

In [42]:
def extraer_mpg(div_automovil):
    mpg_str = div_automovil.find("span", attrs={"class":"mpg"}).text
    try:
        mpg = float(mpg_str.split(' ')[0])
    except ValueError:
        mpg = "NULL"
    return mpg

In [43]:
def extraer_caballos_potencia(div_automovil):
    caballos_potencia_str = div_automovil.find('span', class_='horsepower').text
    try:
        caballos_potencia = float(caballos_potencia_str)
    except ValueError:
        caballos_potencia = "NULL"
    return caballos_potencia

In [44]:
def extraer_desplazamiento(div_automovil_text):
    str_desplazamiento = re.findall(r'.* (\d+.\d+) cubic inches', div_automovil_text)[0]
    desplazamiento = float(str_desplazamiento)
    return desplazamiento

In [45]:
#Nivel de abstracción 3
def extraer_datos(div_automovil):
    nombre = extraer_nombre(div_automovil)
    cilindros = extraer_cilindros(div_automovil)
    peso = extraer_peso(div_automovil)
    territorio, anio = extraer_territorio_anio(div_automovil)
    # territorio = extraer_territorio(div_automovil)
    # anio = extraer_anio(div_automovil)
    aceleracion = extraer_aceleracion(div_automovil)
    mpg = extraer_mpg(div_automovil)
    caballos_potencia = extraer_caballos_potencia(div_automovil)
    desplazamiento = extraer_desplazamiento(div_automovil.text)

    return dict(nombre=nombre,
               cilindros=cilindros,
               peso=peso,
               anio=anio,
               territorio=territorio,
               aceleracion=aceleracion,
               mpg=mpg,
               caballos_potencia=caballos_potencia,
               desplazamiento=desplazamiento)

In [46]:
def obtener_pagina(url):
    respuesta = requests.get(url)
    return BeautifulSoup(respuesta.text, "html.parser" )

In [47]:
def obtener_divs_automoviles(pagina):
    return pagina.body.find_all(name="div", attrs={"class":"car_block"})

In [48]:
def extraer_datos_automoviles(divs_automoviles):
    #Este bloque es equivalente al list comprehension
#     datos_automoviles = []
#     for div_automovil in divs_automoviles:
#         datos_automoviles.add(extraer_datos(div_automovil))

#     return datos_automoviles

    return [extraer_datos(div_automovil) for div_automovil in divs_automoviles]

In [49]:
def guardar_datos_automoviles_en_archivo_csv(datos_automoviles):
    with open("/content/drive/My Drive/Colab Notebooks/SIS-251/02_extracción/02_web_scraping_and_webservices/datos_automoviles.csv", "w", encoding="utf-8", newline="") as file_writer:
        writer = csv.DictWriter(file_writer, fieldnames=datos_automoviles[0].keys())
        writer.writeheader()
        writer.writerows(datos_automoviles)

In [50]:
#Nivel de abstracción 2, imperative shell functional core
def extraer_datos_automoviles_en_archivo_csv(pagina):
    divs_automoviles = obtener_divs_automoviles(pagina)
    datos_automoviles = extraer_datos_automoviles(divs_automoviles)
    guardar_datos_automoviles_en_archivo_csv(datos_automoviles)

In [51]:
#Nivel de abstracción 1
pagina = obtener_pagina("https://mrocabado.github.io/web/auto_mpg.html")
extraer_datos_automoviles_en_archivo_csv(pagina)

## Casos donde la repetición está justificada

**1. Coincidencia temporal, no conceptual**

Cuando dos piezas de código se parecen *ahora*, pero evolucionarán de manera independiente.
Abstraer prematuramente puede crear un acoplamiento artificial que dificulta cambios futuros.

Este es el tipo de duplicación en la clase `com.mindwaresrl.human.CarWebScraping`

```java
    private String extractName(Element div) {
        return div.selectFirst("span.car_name").text();
    }

    private String extractCylinders(Element div) {
        return div.selectFirst("span.cylinders").text();
    }

    private String extractWeight(Element div) {
        return div.selectFirst("span.weight").text();
    }
```

Si la página HTML fuente cambia, la forma de extraer el `name`, `cylinders` o `weight`,
probablemente necesitará cambios distintos.

**2. Contextos muy diferentes**

Código similar en capas o módulos distintos (frontend vs backend, diferentes microservicios). La abstracción compartida
crearía dependencias indeseables entre contextos que deberían ser independientes.

**3. Simplicidad vs complejidad de la abstracción**

Si eliminar la repetición requiere una abstracción compleja, indirecta o difícil de entender, a veces es mejor mantener
código duplicado pero claro. El costo de mantenimiento de la complejidad puede superar el costo de la duplicación.

**4. Casos extremos o validaciones específicas**

Validaciones o manejo de errores que parecen similares, pero responden a requisitos de negocio distintos.
Combinarlas puede hacer que cambios en un caso afecten inadvertidamente a otros.

**5. Performance crítica**

En código de alto rendimiento, la duplicación puede evitar abstracciones que agreguen overhead innecesario.

**6. Regla de los tres**

Esperar a tener 2-3 instancias de repetición antes de abstraer. Con un solo caso de duplicación, aún no sabes el patrón real.

### Recomendación práctica
El error común es ver código similar y pensar "esto está duplicado, debo abstraerlo YA". Pero la similitud *temporal*
no implica que sea el mismo concepto. Muchas veces dos cosas se parecen al inicio, pero divergen naturalmente con los
requisitos del negocio.

Es mejor tolerar algo de duplicación inicial y esperar a ver cómo evoluciona el código antes de crear una abstracción.
Una vez que tienes 3+ casos y entiendes el patrón real, entonces abstraes con confianza.

Como dice el principio: **"Duplicación es mejor que la abstracción incorrecta"**, el código duplicado es evidente
y fácil de cambiar; una mala abstracción contamina todo el código base y es difícil de revertir.

# Ejercicios

Apoyándose en Gemini:
- Agregar Type Hits a todas las funciones que no los tengan
- Averiguar para qué el sirve los parámetros especiales `/` y `*`?